In [1]:
!pip install python-docx lxml

from google.colab import files
import docx
import zipfile
import re
from lxml import etree

# Subir archivo
uploaded = files.upload()
docx_path = next(iter(uploaded))

# Leer relaciones de imágenes
with zipfile.ZipFile(docx_path) as docx_zip:
    rels_xml = docx_zip.read('word/_rels/document.xml.rels')
    rels_root = etree.fromstring(rels_xml)
    rels_map = {}
    for rel in rels_root.findall('.//{http://schemas.openxmlformats.org/package/2006/relationships}Relationship'):
        rId = rel.get('Id')
        target = rel.get('Target')
        rels_map[rId] = target

# Cargar documento
doc = docx.Document(docx_path)

# Crear estructura raíz
NSMAP = {"xlink": "http://www.w3.org/1999/xlink"}
article = etree.Element("article")
body = etree.SubElement(article, "body", nsmap=NSMAP)

# Variables de control
section_numbers = []
current_paragraph_container = body
in_list = False
current_list = None
list_type = None
table_count = 0
fig_counter = 1

# Recorrer elementos
elements = list(doc.element.body.iterchildren())
for idx, el in enumerate(elements):
    if el.tag.endswith('tbl'):
        # === TABLA ===
        table_data = docx.table.Table(el, doc)
        table_count += 1

        table_wrap = etree.SubElement(current_paragraph_container, "table-wrap", id=f"tabl{table_count}")
        label = etree.SubElement(table_wrap, "label")
        label.text = f"Tabla {table_count}"
        caption = etree.SubElement(table_wrap, "caption")
        title = etree.SubElement(caption, "title")
        title.text = f"Tabla {table_count}. Título de ejemplo"

        table = etree.SubElement(table_wrap, "table")
        thead = etree.SubElement(table, "thead")
        tr_head = etree.SubElement(thead, "tr")

        for i, row in enumerate(table_data.rows):
            if i == 0:
                for cell in row.cells:
                    th = etree.SubElement(tr_head, "th", style="text-align: center;")
                    bold = etree.SubElement(th, "bold")
                    bold.text = cell.text
            else:
                if i == 1:
                    tbody = etree.SubElement(table, "tbody")
                tr = etree.SubElement(tbody, "tr")
                for cell in row.cells:
                    td = etree.SubElement(tr, "td", style="text-align: center;")
                    td.text = cell.text

        in_list = False
        current_list = None

    elif el.tag.endswith('p'):
        # === PÁRRAFO ===
        p = docx.text.paragraph.Paragraph(el, doc)
        if not p.style:
            continue
        estilo = p.style.name.lower()
        texto = p.text.strip()

        # Detectar imagen
        if 'graphic' in el.xml:
            fig = etree.SubElement(current_paragraph_container, "fig", id=f"fig{fig_counter}")
            caption_text = ""

            # Buscar siguiente párrafo como caption
            if idx + 1 < len(elements):
                next_el = elements[idx + 1]
                if next_el.tag.endswith('p'):
                    next_p = docx.text.paragraph.Paragraph(next_el, doc)
                    if next_p.text.strip():
                        caption_text = next_p.text.strip()

            caption = etree.SubElement(fig, "caption")
            p_caption = etree.SubElement(caption, "p")
            p_caption.text = caption_text if caption_text else f"Figura {fig_counter}. Sin caption detectado"

            rId_search = re.search(r'r:embed="(rId\d+)"', el.xml)
            if rId_search:
                rId = rId_search.group(1)
                href = rels_map.get(rId, f"image{fig_counter}.jpeg")
            else:
                href = f"image{fig_counter}.jpeg"

            graphic = etree.SubElement(fig, "graphic", attrib={"{http://www.w3.org/1999/xlink}href": href})
            fig_counter += 1

            in_list = False
            current_list = None
            continue

        # --- TÍTULOS ---
        if "heading" in estilo and re.search(r"\d+", estilo):
            nivel = int(re.search(r"\d+", estilo).group()) - 2
            if nivel < 0:
                continue

            while len(section_numbers) <= nivel:
                section_numbers.append(0)
            section_numbers[nivel] += 1
            section_numbers = section_numbers[:nivel + 1]

            section_id = ".".join(map(str, section_numbers))
            sec_elem = etree.Element("sec", id=f"sec-{section_id}")
            sec_title = etree.SubElement(sec_elem, "title")
            sec_title.text = texto

            parent = body
            for i in range(len(section_numbers) - 1):
                if len(parent) == 0 or parent[-1].tag != "sec":
                    parent = etree.SubElement(parent, "sec")
                else:
                    parent = parent[-1]
            parent.append(sec_elem)
            current_paragraph_container = sec_elem

            in_list = False
            current_list = None

        # --- LISTAS ---
        elif "list" in estilo or re.match(r"^(\d+[\.\)]|[•\-])\s+", texto):
            if not in_list:
                list_type = "ordered" if re.match(r"^\d+[\.\)]\s+", texto) else "bullet"
                current_list = etree.SubElement(current_paragraph_container, "list", attrib={"list-type": list_type})
                in_list = True

            list_item = etree.SubElement(current_list, "list-item")
            p_elem = etree.SubElement(list_item, "p")
            p_elem.text = re.sub(r"^(\d+[\.\)]|[•\-])\s+", "", texto)

        # --- PÁRRAFOS NORMALES ---
        else:
            if in_list:
                in_list = False
                current_list = None
            p_elem = etree.SubElement(current_paragraph_container, "p")
            p_elem.text = texto

# Mostrar resultado final
xml_str = etree.tostring(article, pretty_print=True, encoding="unicode")
print(xml_str)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.7 MB/s eta 0:00:00


KeyboardInterrupt: 